# VQA Múa Lân — Train & Eval (A1, A2, A3)

Huấn luyện và đánh giá **3 cấu hình decoder** trên cùng pipeline:

| Run | Decoder       | Norm      | FFN        | Ghi chú |
|-----|---------------|-----------|------------|---------|
| **A1** | LSTM        | —         | —          | Baseline rời (RNN cổ điển) |
| **A2** | Transformer | LayerNorm | VanillaFFN | Transformer cổ điển (Attention Is All You Need) |
| **A3** | Transformer | RMSNorm   | SwiGLU     | Modern stack (LLaMA / Gemma / Qwen) |

Encoder cố định: SigLIP2-B/16 (layer −2, frozen) + PhoBERT-v2 (mean 4 last layers, frozen) + Cross-Attention Fusion.
Loss: `CrossEntropy(ignore_index=PAD, label_smoothing=0.1)`.

Notebook đặt ở project root; mọi class import trực tiếp từ `src/` — không lặp code.

## 0. Kaggle Setup

In [1]:
!git clone https://github.com/KhanhNamYeh/qa-domain

Cloning into 'qa-domain'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 64 (delta 20), reused 52 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 199.06 KiB | 3.62 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
!pwd

/kaggle/working


In [3]:
!ls

qa-domain


## 1. Setup

In [4]:
import os, sys, time, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# PROJECT_ROOT = os.path.abspath('.')
PROJECT_ROOT = os.path.abspath('/kaggle/working/qa-domain')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ModelConfig, TrainConfig
from src.build  import (build_tokenizer_and_processor, resolve_special_ids,
                        build_loaders, build_model)
from src.training import Trainer, Evaluator
from src.metrics  import ExactMatch, BLEUScore, METEORScore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('project_root =', PROJECT_ROOT)
print('device       =', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

project_root = /kaggle/working/qa-domain
device       = cuda


## 2. Tokenizer & DataLoaders (dùng chung cho 3 run)

In [5]:
_probe_cfg = ModelConfig()
tokenizer, image_processor = build_tokenizer_and_processor(_probe_cfg)
_probe_cfg = resolve_special_ids(tokenizer, _probe_cfg)

BATCH_SIZE = 16
EPOCHS     = 8
LR         = 3e-4

# Đường dẫn dữ liệu phải truyền tường minh — TrainConfig không có default cho mục này.
TRAIN_JSON = os.path.join(PROJECT_ROOT, 'qa_data', 'train.json')
VAL_JSON   = os.path.join(PROJECT_ROOT, 'qa_data', 'val.json')
TEST_JSON  = os.path.join(PROJECT_ROOT, 'qa_data', 'test.json')
IMAGE_ROOT = os.path.join('/kaggle/input/datasets/namkhn/qa-domain', 'images')

shared_train_cfg = TrainConfig(
    train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
    image_root=IMAGE_ROOT,
    batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=0,
    ckpt_dir=os.path.join(PROJECT_ROOT, 'checkpoints'),
    log_dir=os.path.join(PROJECT_ROOT, 'logs'),
    eval_every=1, save_every=EPOCHS,
)
train_loader, val_loader, test_loader = build_loaders(
    _probe_cfg, shared_train_cfg, tokenizer, image_processor)
print(f'train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  test={len(test_loader.dataset)}')

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


train=9879  val=1235  test=1235


## 3. Trainer-with-history

Bọc lại `Trainer` để lưu `train_loss` và mọi metric val theo từng epoch — phục vụ vẽ biểu đồ line.

In [6]:
class TrainerWithHistory(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.history = {'train_loss': [], 'val': {}}
    def fit(self):
        cfg = self.train_cfg
        for epoch in range(cfg.epochs):
            t0 = time.time()
            tl = self.train_one_epoch(epoch)
            self.history['train_loss'].append(tl)
            log = f'epoch={epoch+1}/{cfg.epochs} train_loss={tl:.4f} time={time.time()-t0:.1f}s'
            if self.evaluator is not None and self.val_loader is not None:
                res = self.evaluator.evaluate(self.model, self.val_loader)
                for k, v in res.items():
                    self.history['val'].setdefault(k, []).append(v)
                log += '  ' + ' '.join(f'val_{k}={v:.4f}' for k, v in res.items())
            self._log(log)
        self._save_ckpt('final')
        return self.history

In [7]:
def run_config(tag, decoder_type, norm_type='layernorm', ffn_type='vanilla'):
    """`norm_type` / `ffn_type` chỉ ảnh hưởng khi decoder_type='transformer';
    với 'lstm' chúng bị bỏ qua nên có thể không truyền."""
    print(f'\n========== {tag} ==========')
    mcfg = ModelConfig(decoder_type=decoder_type, norm_type=norm_type, ffn_type=ffn_type)
    mcfg = resolve_special_ids(tokenizer, mcfg)
    tcfg = TrainConfig(
        train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
        image_root=IMAGE_ROOT,
        batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=0,
        ckpt_dir=os.path.join(PROJECT_ROOT, 'checkpoints'),
        log_dir=os.path.join(PROJECT_ROOT, 'logs'),
        eval_every=1, save_every=EPOCHS, run_name=tag,
    )
    model = build_model(mcfg)
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    evaluator = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=mcfg.bos_id, eos_id=mcfg.eos_id, pad_id=mcfg.pad_id,
        max_len=mcfg.max_answer_len, beam_size=1,
    )
    trainer = TrainerWithHistory(
        model=model, train_loader=train_loader, val_loader=val_loader,
        train_cfg=tcfg, model_cfg=mcfg, evaluator=evaluator,
    )
    history = trainer.fit()
    return history, trainer, evaluator, model

## 4. Train A1 — LSTM

In [8]:
hist_A1, trainer_A1, evaluator_A1, model_A1 = run_config(
    'A1_lstm', decoder_type='lstm')


========== A1_lstm ==========


config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

[2026-05-04 03:37:01] epoch=1/8 train_loss=3.0376 time=344.7s  val_exact_match=0.1976 val_bleu=0.5307 val_meteor=0.6853
[2026-05-04 03:43:15] epoch=2/8 train_loss=2.0799 time=338.4s  val_exact_match=0.2453 val_bleu=0.5853 val_meteor=0.7350
[2026-05-04 03:49:41] epoch=3/8 train_loss=1.9539 time=348.0s  val_exact_match=0.2794 val_bleu=0.6055 val_meteor=0.7503
[2026-05-04 03:56:06] epoch=4/8 train_loss=1.8856 time=347.6s  val_exact_match=0.3158 val_bleu=0.6217 val_meteor=0.7621
[2026-05-04 04:02:40] epoch=5/8 train_loss=1.8386 time=354.0s  val_exact_match=0.3312 val_bleu=0.6263 val_meteor=0.7655
[2026-05-04 04:09:59] epoch=6/8 train_loss=1.8029 time=391.4s  val_exact_match=0.3522 val_bleu=0.6353 val_meteor=0.7741
[2026-05-04 04:16:35] epoch=7/8 train_loss=1.7757 time=359.1s  val_exact_match=0.3506 val_bleu=0.6474 val_meteor=0.7793
[2026-05-04 04:22:58] epoch=8/8 train_loss=1.7520 time=345.6s  val_exact_match=0.3312 val_bleu=0.6325 val_meteor=0.7716
[2026-05-04 04:23:00] Saved checkpoint -

## 5. Train A2 — Transformer + LayerNorm + VanillaFFN

In [9]:
hist_A2, trainer_A2, evaluator_A2, model_A2 = run_config(
    'A2_transformer_vanilla', decoder_type='transformer', norm_type='layernorm', ffn_type='vanilla')


========== A2_transformer_vanilla ==========


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[2026-05-04 04:29:07] epoch=1/8 train_loss=2.4492 time=321.8s  val_exact_match=0.2494 val_bleu=0.5745 val_meteor=0.7227
[2026-05-04 04:35:14] epoch=2/8 train_loss=1.9298 time=321.8s  val_exact_match=0.3028 val_bleu=0.6204 val_meteor=0.7612
[2026-05-04 04:41:23] epoch=3/8 train_loss=1.8452 time=324.2s  val_exact_match=0.3263 val_bleu=0.6296 val_meteor=0.7708
[2026-05-04 04:47:37] epoch=4/8 train_loss=1.7978 time=328.4s  val_exact_match=0.3360 val_bleu=0.6381 val_meteor=0.7726
[2026-05-04 04:53:43] epoch=5/8 train_loss=1.7631 time=322.3s  val_exact_match=0.3506 val_bleu=0.6485 val_meteor=0.7785
[2026-05-04 04:59:55] epoch=6/8 train_loss=1.7408 time=326.8s  val_exact_match=0.3457 val_bleu=0.6440 val_meteor=0.7750
[2026-05-04 05:06:09] epoch=7/8 train_loss=1.7193 time=326.5s  val_exact_match=0.3684 val_bleu=0.6577 val_meteor=0.7872
[2026-05-04 05:12:39] epoch=8/8 train_loss=1.7042 time=326.8s  val_exact_match=0.3490 val_bleu=0.6440 val_meteor=0.7844
[2026-05-04 05:12:41] Saved checkpoint -

## 6. Train A3 — Transformer + RMSNorm + SwiGLU

In [ ]:
hist_A3, trainer_A3, evaluator_A3, model_A3 = run_config(
    'A3_transformer_modern', decoder_type='transformer', norm_type='rmsnorm', ffn_type='swiglu')


========== A3_transformer_modern ==========


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7. So sánh — biểu đồ line

In [ ]:
def plot_compare(histories, keys, title_prefix=''):
    n = len(keys)
    fig, axes = plt.subplots(1, n, figsize=(5.2*n, 4))
    if n == 1: axes = [axes]
    for ax, key in zip(axes, keys):
        for tag, h in histories.items():
            ys = h['train_loss'] if key == 'train_loss' else h['val'].get(key, [])
            xs = list(range(1, len(ys)+1))
            ax.plot(xs, ys, marker='o', label=tag)
        ax.set_title(f'{title_prefix}{key}')
        ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

histories = {
    'A1_lstm':                hist_A1,
    'A2_transformer_vanilla': hist_A2,
    'A3_transformer_modern':  hist_A3,
}
plot_compare(histories, ['train_loss'])
plot_compare(histories, ['exact_match', 'bleu', 'meteor'], title_prefix='val_')

## 8. Đánh giá cuối trên test split (Exact Match, BLEU, METEOR)

In [ ]:
test_results = {}
for tag, model in [('A1_lstm', model_A1),
                   ('A2_transformer_vanilla', model_A2),
                   ('A3_transformer_modern',  model_A3)]:
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    ev = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=_probe_cfg.bos_id, eos_id=_probe_cfg.eos_id, pad_id=_probe_cfg.pad_id,
        max_len=_probe_cfg.max_answer_len, beam_size=1,
    )
    test_results[tag] = ev.evaluate(model, test_loader)
    print(tag, test_results[tag])

## 9. Lưu lại history + test results

In [ ]:
log_dir = os.path.join(PROJECT_ROOT, 'logs')
os.makedirs(log_dir, exist_ok=True)
with open(os.path.join(log_dir, 'train_eval_A_history.json'), 'w', encoding='utf-8') as f:
    json.dump({'history': histories, 'test': test_results}, f, ensure_ascii=False, indent=2)
print('saved -> logs/train_eval_A_history.json')